In [0]:
import os, sys
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), "src"))
from pyspark.sql import functions as F
from lh import common, silver

# t30_new_columns: a new source column (config/silver.yml `from_rescued`, e.g. patients.preferred_language and
# loans.channel from 15 Oct 2026) reaches Bronze, Silver and Gold, and nothing older is changed.
# Read-only. Same file in both repos. Run from your dev Git folder; env widget default dev
# (in test/prod only the clear group can see Bronze _rescued_data). After running: discard the notebook changes.
# FAIL = the column is wrong somewhere; WARN = no row from the new-column date has been loaded yet.
dbutils.widgets.dropdown("env", "dev", ["dev", "test", "prod"])
cfg = common.load_config(dbutils.widgets.get("env"))
cat = cfg["catalog"]
since = str(cfg["new_column_date"])                    # config/company.yml: first day the source sends it
results = []


def check(level, name, ok, detail=""):
    status = "PASS" if ok else level
    results.append(status)
    print(f"{status:4} {name}", detail)


new = {t: s["from_rescued"] for t, s in silver.load_silver().items() if s.get("from_rescued")}
check("FAIL", "config/silver.yml names at least one new column (from_rescued)", bool(new), new)
for table, cols in new.items():
    spec = silver.load_silver()[table]
    bronze = spark.table(f"{cat}.bronze.{table}")
    stable = spark.table(f"{cat}.silver.{table}")
    for c in cols:
        b = bronze.agg(F.count_if(F.col("_business_date") >= since).alias("rows_new"),
                       F.count_if((F.col("_business_date") >= since) & silver.source_col(bronze, spec, c).isNotNull()).alias("with"),
                       F.count_if((F.col("_business_date") < since) & silver.source_col(bronze, spec, c).isNotNull()).alias("early")).first()
        if b["rows_new"] == 0:
            check("WARN", f"bronze.{table}: rows from {since} loaded", False, "(none yet: run the simulator and the daily job)")
            continue
        check("FAIL", f"bronze.{table}.{c}: in every row from {since}, in none before",
              b["with"] == b["rows_new"] and b["early"] == 0, (b["with"], b["rows_new"], b["early"]))
        has = c in stable.columns
        check("FAIL", f"silver.{table}: has column {c}", has)
        if not has:
            continue
        s = stable.agg(F.count_if(F.col("_business_date") >= since).alias("rows_new"),
                       F.count_if((F.col("_business_date") >= since) & F.col(c).isNull()).alias("missing"),
                       F.count_if((F.col("_business_date") < since) & F.col(c).isNotNull()).alias("early"),
                       F.countDistinct(c).alias("values")).first()
        check("FAIL", f"silver.{table}.{c}: filled for rows from {since}, NULL before",
              s["rows_new"] > 0 and s["missing"] == 0 and s["early"] == 0, tuple(s))
        gold = [r.table_name for r in spark.sql(f"""SELECT table_name FROM {cat}.information_schema.columns
                 WHERE table_schema = 'gold' AND column_name = '{c}' AND table_name NOT LIKE 'v\\_%'""").collect()]
        check("FAIL", f"gold: a dimension carries {c}", bool(gold), gold)
        for g in gold:
            n = spark.sql(f"SELECT count_if(`{c}` IS NOT NULL) FROM {cat}.gold.`{g}`").first()[0]
            check("FAIL", f"gold.{g}.{c}: has values", n > 0, n)

changes = spark.sql(f"""SELECT count(*) FROM {cat}.ops.audit_runs
                        WHERE job = 'silver.schema_change' AND status = 'success'""").first()[0]
check("WARN", "schema change recorded in ops.audit_runs (only where the table existed before)", changes > 0, changes)

print(f"\n{cat}: {results.count('PASS')}/{len(results)} PASS, {results.count('WARN')} WARN, {results.count('FAIL')} FAIL")
assert "FAIL" not in results, "t30_new_columns failed"